# Agentic segmentation — VLM-only variant

Gemini 3.1 Pro replaces **both** Grounding DINO (detection) and its own role as reviewer.
Pipeline: `user text → Gemini refine → Gemini detect_boxes → MedSAM → Gemini analyze → loop`.

Outputs go under `data/agent_{runs,results,cache}_vlm/` so they don't collide with the DINO+Gemini baseline.

In [ ]:
# === Colab bootstrap ================================================
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, subprocess, sys
    REPO = '/content/INF8225_Projet'
    if not os.path.isdir(REPO):
        subprocess.check_call(['git', 'clone', '--depth', '1',
            'https://github.com/Azcatchi17/INF8225_Projet.git', REPO])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from colab.setup import setup
    setup()

In [ ]:
# === Load models ====================================================
# NB: DINO is NOT loaded in the VLM variant — saves ~2 GB VRAM.
from pipeline.models import get_medsam_model, get_gemma_client

_ = get_medsam_model()
gemma = get_gemma_client()
print('models ready (Gemini + MedSAM, no DINO)')

In [ ]:
# === Gemini detection smoke test ====================================
from pipeline_vlm import config as vlm_cfg

sample_img = next(vlm_cfg.KVASIR_IMAGES.glob('*.jpg'))
boxes = gemma.detect_boxes(str(sample_img), 'polyp',
                           score_threshold=vlm_cfg.VLM_SCORE_THRESHOLD,
                           top_k=vlm_cfg.VLM_TOP_K)
for b in boxes:
    print(f'score={b.score:.2f}  xyxy={[round(v,1) for v in b.xyxy]}')

In [ ]:
# === Single-image demo ==============================================
import matplotlib.pyplot as plt
import numpy as np
from skimage import io
from pipeline_vlm import config as vlm_cfg
from pipeline_vlm.agent import run_agent

demo_file = next(vlm_cfg.KVASIR_IMAGES.glob('*.jpg'))
gt_path = vlm_cfg.KVASIR_MASKS / demo_file.name
gt = None
if gt_path.exists():
    m = io.imread(gt_path)
    gt = (m[..., 0] if m.ndim == 3 else m) > 0
    gt = gt.astype(np.uint8)

state = run_agent(str(demo_file), 'find the polyp', gt_mask=gt)
print(f'run_id         : {state.run_id}')
print(f'refined prompt : {state.refined_prompt}')
print(f'stop reason    : {state.stop_reason}')
best = state.best_iter()
print(f'best iter      : {best.iteration if best else "—"}')
for it in state.iterations:
    act = it.gemma_action.action if it.gemma_action else '—'
    print(f'  iter {it.iteration}: action={act:<14} '
          f'dice={it.dice_vs_gt} area_pct={it.metrics.area_pct:.3f}')

n = 1 + len(state.iterations)
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
axes[0].imshow(state.image_np); axes[0].set_title('image'); axes[0].axis('off')
for i, it in enumerate(state.iterations, start=1):
    axes[i].imshow(state.image_np)
    axes[i].imshow(it.mask, alpha=0.45, cmap='Reds')
    marker = ' ★' if best and it.iteration == best.iteration else ''
    axes[i].set_title(f'iter {it.iteration}{marker}'
                      + (f'  DICE={it.dice_vs_gt:.2f}' if it.dice_vs_gt is not None else ''))
    axes[i].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# === VLM batch evaluation ===========================================
from pipeline_vlm.eval import run_batch

df = run_batch(n=20)
df.describe()[['first_iter_dice', 'last_iter_dice', 'final_dice', 'n_iterations']]

In [ ]:
# === VLM one-shot vs VLM agentic ====================================
from pipeline_vlm.eval import compare_oneshot_vs_agentic

cmp_df = compare_oneshot_vs_agentic(n=50)
print(cmp_df[['one_shot_dice', 'agentic_dice', 'dice_delta']].describe())